# 06 — Train a Multi-Layer Perceptron From Scratch

Now we assemble the entire learning loop: initialize → forward → loss → backward → update → repeat.

We use XOR because a single linear neuron cannot solve it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4,suppress=True)

## 1. XOR dataset

In [ ]:
X=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=np.array([[0.],[1.],[1.],[0.]])
plt.figure(figsize=(5,5)); plt.scatter(X[:,0],X[:,1],c=y.ravel(),s=180)
for i,(a,b) in enumerate(X): plt.text(a+.03,b+.03,f'y={int(y[i,0])}')
plt.xlim(-.2,1.2); plt.ylim(-.2,1.2); plt.grid(alpha=.25); plt.title('XOR is not linearly separable'); plt.show()

## 2. Implement a 2 → 4 → 1 MLP

In [ ]:
rng=np.random.default_rng(42)
W1=rng.normal(0,.8,(2,4)); b1=np.zeros((1,4))
W2=rng.normal(0,.8,(4,1)); b2=np.zeros((1,1))
def sigmoid(z): return 1/(1+np.exp(-z))
def forward(X):
    z1=X@W1+b1; a1=np.tanh(z1); z2=a1@W2+b2; yhat=sigmoid(z2); return z1,a1,z2,yhat
def bce(y,yhat):
    eps=1e-9; return -np.mean(y*np.log(yhat+eps)+(1-y)*np.log(1-yhat+eps))

## 3. Manual backpropagation and parameter updates

In [ ]:
lr=.8; history=[]
for epoch in range(5000):
    z1,a1,z2,yhat=forward(X); loss=bce(y,yhat)
    dz2=(yhat-y)/len(X); dW2=a1.T@dz2; db2=np.sum(dz2,axis=0,keepdims=True)
    da1=dz2@W2.T; dz1=da1*(1-a1**2); dW1=X.T@dz1; db1=np.sum(dz1,axis=0,keepdims=True)
    W2-=lr*dW2; b2-=lr*db2; W1-=lr*dW1; b1-=lr*db1
    if epoch%25==0: history.append(loss)
print('final loss:',loss)
print('probabilities:',np.round(yhat.ravel(),4))
print('classes:',(yhat>.5).astype(int).ravel())

## 4. Watch the loss fall

In [ ]:
plt.figure(figsize=(9,4)); plt.plot(np.arange(len(history))*25,history); plt.xlabel('epoch'); plt.ylabel('binary cross entropy'); plt.title('Training curve'); plt.grid(alpha=.25); plt.show()

## 5. Visualize the nonlinear decision surface the network learned

In [ ]:
xx,yy=np.meshgrid(np.linspace(-.5,1.5,300),np.linspace(-.5,1.5,300))
grid=np.c_[xx.ravel(),yy.ravel()]
_,_,_,pp=forward(grid); P=pp.reshape(xx.shape)
plt.figure(figsize=(7,6)); plt.contourf(xx,yy,P,levels=30,alpha=.7); plt.contour(xx,yy,P,levels=[.5],linewidths=2); plt.scatter(X[:,0],X[:,1],c=y.ravel(),s=180,edgecolors='black'); plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Nonlinear boundary learned by the hidden layer'); plt.show()

This notebook contains the complete neural-network learning mechanism without autograd or a deep-learning framework.